# Module 6: Connecting tools through the platform

So far the agent's tools are built into its own code. In this module you point the agent at an MCP gateway on the platform, then add a new tool there without changing the agent. MCP, the Model Context Protocol, is an open standard for offering tools to agents.

In [ ]:
# Load the lab's notebook helpers and work from the agent's folder.
import labnb as lab
%cd ../agent

## Start from module 6

This cell puts the agent back where module 6 starts. It copies any edits you made to
`agent/.reset-backup/` first, so nothing you wrote is lost.

In [ ]:
# Put the agent in module 6's starting state.
!python reset.py 6

## 1. How the agent gets its tools today

The agent's tools are written in [`tools.py`](../agent/tools.py), the file of tool functions it runs. [`config.toml`](../agent/config.toml), the settings file the agent reads on every run, gives each tool its own address and credential. The next cell prints the `[tools]` part of it.

In [ ]:
# Print the [tools] settings from config.toml: gateway_url, then each tool's own entry.
import json
tools = lab.config()["tools"]
print("gateway_url:", repr(tools.get("gateway_url")))
print(json.dumps({k: v for k, v in tools.items() if k != "gateway_url"}, indent=2))

`gateway_url` is empty, so the agent uses its own tools. Each address here is a local file in `agent/fixtures/`, and nothing checks the credentials: `credential_env` is the name of the environment variable a real credential would come from. In production each address would be a service URL and each credential a real token. `search_bulletins`, the tool you added in module 5, has no entry because it takes its database settings from the standard PostgreSQL environment variables (see [`retrieval.py`](../agent/retrieval.py)).

Run module 3's question on the agent's own tools, so you have a result to compare with later:

In [ ]:
# Ask the agent the lab's question 'm3':
#   "Can we get a replacement coolant pump for a K-400 to the Denver depot this week?"
!python agent.py --q m3
# Show a table of this notebook's runs, one row each:
#   turns: how many times the agent called the model
#   ended by: what stopped the run
#   tools run: the tools the agent ran, and each result's category
lab.evidence(columns=("turns", "ended by", "tools run"))

In this lab the agent runs straight from source in your workbench. In production it would run as a container image, so adding a tool would mean editing the agent, rebuilding that image and redeploying it in every environment it runs in. Every tool change would be an application release.

```mermaid
flowchart TB
  subgraph Today["Today"]
    A1["agent"] --> T1["tools.py: find_part, check_stock, search_bulletins"]
  end
  subgraph Platform["With MCP Gateway"]
    A2["agent"] -- "one gateway_url" --> G["MCP Gateway 0.7.1 (Technology Preview)"]
    G --> S1["The lab's tool server"]
    G -. "registered by the platform team" .-> S2["estimate_delivery"]
  end
  Today ~~~ Platform
```

## 2. Point the agent at the gateway

An MCP gateway sits in front of tool servers and gives every agent one address to find and call their tools. Behind the lab's gateway is the lab's tool server, [`lab-tool-server.py`](../automation/gitops/bootstrap-infra/files/lab-tool-server.py): a small service that runs on the cluster, outside your workbench, and is shared by every participant. It serves the same `find_part`, `check_stock` and `search_bulletins` as your `tools.py`, with the lab's own fixes from modules 4 and 5 (the files in `agent/solutions/`), not your copies. Its `search_bulletins` reads its own read-only copy of the service bulletins.

The lab runs MCP Gateway 0.7.1, a Technology Preview. A generally available version ships in [Red Hat Connectivity Link 1.5](https://www.redhat.com/en/blog/model-context-protocol-gateway-red-hat-openshift-now-generally-available).

In [ ]:
# Print the lab MCP Gateway's address, which your workbench was given when it was set up.
print(lab.gateway_url() or "This environment has no lab MCP Gateway (LAB_GATEWAY_URL isn't set).")

Click [`config.toml`](../agent/config.toml) to open it in a new tab in JupyterLab. Under `[tools]`, paste the address between the quotes on the `gateway_url = ""` line, so it reads `gateway_url = "https://..."`. Keep the quotes: without them the file isn't valid TOML and the next cells fail. Leave the `[tools.find_part]` and `[tools.check_stock]` blocks alone, even though the comment above them says module 6 replaces them. Once `gateway_url` is set, the agent ignores them. Save with Ctrl+S, then come back to this notebook's tab.

That's the only change to the agent. The agent picks its tools in [`toolbox.py`](../agent/toolbox.py), a short file that chooses where the tools come from on every run. When `gateway_url` is set, it asks the gateway for the tool list and sends every tool call there. When it's empty, it uses `tools.py`.

In [ ]:
# Show open_toolbox from toolbox.py, the function that reads gateway_url.
lab.show_code("toolbox.py", "open_toolbox")

Run the next cell to check what the agent will use. You should see your gateway address in quotes. If you see `''`, save `config.toml` and run the cell again. Don't go on until it shows the address.

In [ ]:
# Print the gateway_url the agent will use, read from config.toml as it's saved now.
gateway = lab.config()["tools"].get("gateway_url")
print("gateway_url:", repr(gateway))
if not gateway:
    print("That's empty. Paste the gateway address into config.toml, save it, and run this cell again.")

## 3. Nothing changes about what the agent does (stretch goal)

Skip this step if you're short of time. Here you ask module 3's question again, this time through the gateway, and check that the agent finds the same facts.

In [ ]:
# Ask module 3's question again, through the gateway this time (only once gateway_url is set).
if lab.config()["tools"].get("gateway_url"):
    # Ask the agent the lab's question 'm3':
    #   "Can we get a replacement coolant pump for a K-400 to the Denver depot this week?"
    lab.sh("python agent.py --q m3")
    # Show a table of this notebook's runs, one row each:
    #   turns: how many times the agent called the model
    #   ended by: what stopped the run
    #   tools run: the tools the agent ran, and each result's category
    lab.evidence(columns=("turns", "ended by", "tools run"))
else:
    print("gateway_url is empty, so the agent still uses its own tools.")

The table has two rows now. The first is the run on the agent's own tools. The second went through the gateway, and its tool names start with `kestrel_`. The gateway adds that prefix to every tool from the lab's tool server, so tools from different servers can't clash. The agent reads the names from the gateway at the start of each run, so nothing in the agent has to change.

Compare the facts. Both runs should find the pump `KI-CP-40817`, with quantity 3 and a lead time of 1 day at Denver. The number of turns can differ, and the model may also check the seal kit, `KI-SK-40233`. Both are fine.

The agent didn't carry a tool list this time. It asked the gateway for one at the start of the run. The next cell prints that list.

In [ ]:
# List the tools the gateway offers the agent right now.
if lab.config()["tools"].get("gateway_url"):
    lab.tool_list()
else:
    print("Set gateway_url first.")

There are seven. The first four, `discover_tools`, `filter_tools_by_tags`, `list_tags` and `select_tools`, come with the gateway itself, which is why they have no `kestrel_` prefix. Going by their names, they help an agent search a large tool catalog. This agent doesn't need them, and the model didn't call them in the lab's test runs. The last three are the Kestrel tools. There's no delivery tool.

## 4. A new tool, without touching the agent

The gateway doesn't offer a delivery tool yet. Ask the delivery question first, to see what the agent does without one.

The question says "that pump", but each run starts fresh, so the agent has no earlier pump to refer to. That doesn't matter here: the transit time depends only on where the shipment leaves from and where it goes.

In [ ]:
# Ask the delivery question before the delivery tool exists (only once gateway_url is set).
if lab.config()["tools"].get("gateway_url"):
    # Ask the agent the lab's question 'm6':
    #   "If we ship that pump from Denver to the customer site in Boulder, when does it arrive?"
    lab.sh("python agent.py --q m6")
    # Show each tool the newest run called, with its arguments and exactly what it returned.
    lab.tool_results()
else:
    print("Set gateway_url first.")

**Your read:** the run called no tools, and the answer says the tools don't cover shipping. Is that another quiet failure?

<details><summary>Check your answer</summary>

No, that's the right behaviour. With no delivery tool, the agent says it can't answer instead of making up a transit time. If your run gave a number of days anyway, the model guessed it, which is the kind of unsupported answer module 5 dealt with.

</details>

Now register `estimate_delivery` at the gateway. For this step you're playing the platform team. The tool's code already exists: it's [`estimate_delivery.py`](../agent/estimate_delivery.py), which the lab's tool server loads. The agent never imports it. The tool server lists the tool only for a project that has registered it, and registering means adding one key, `estimate_delivery`, to a ConfigMap called `kestrel-tool-registrations` in your own project. The next cell does that for you. Nothing in the agent changes, and the agent isn't rebuilt or restarted. Only your catalog changes. Other participants' catalogs don't. To see the ConfigMap, open the OpenShift console tab, go to your project, then Workloads > ConfigMaps.

The tool server checks registrations again every few seconds. The cell prints the tool list after registering. If `kestrel_estimate_delivery` isn't at the end of it, wait a few seconds and run the cell again. Registering twice does no harm. Don't go on until it's listed. The cell registers nothing until `gateway_url` is set, so the run above stays a true "before".

If you've done this module before, `estimate_delivery` is already registered. A registration lasts until someone removes it, and the reset at the top of this notebook doesn't remove it. In that case the run above already found the tool and answered, which is expected. Removing the `estimate_delivery` key from the ConfigMap unregisters the tool. Register it again before module 7, which needs it.

In [ ]:
# Register estimate_delivery for your project, then list the tools the gateway offers now.
# It does nothing until gateway_url is set, so the delivery run above stays a true "before".
if lab.config()["tools"].get("gateway_url"):
    lab.register_tool("estimate_delivery")
    lab.tool_list()
else:
    print("Set gateway_url first. Nothing was registered.")

Ask the delivery question again. The agent is unchanged. It picks up the new tool when it asks the gateway for its tool list at the start of the run.

In [ ]:
# Ask the delivery question again, now that the tool is registered (only once gateway_url is set).
if lab.config()["tools"].get("gateway_url"):
    # Ask the agent the lab's question 'm6':
    #   "If we ship that pump from Denver to the customer site in Boulder, when does it arrive?"
    lab.sh("python agent.py --q m6")
    # Show each tool the newest run called, with its arguments and exactly what it returned.
    lab.tool_results()
    # A link to the newest run's trace in MLflow.
    lab.trace_link()
else:
    print("Set gateway_url first.")

**This is the module's success signal:** the agent found and used `estimate_delivery`, listed as `kestrel_estimate_delivery`, a tool it didn't have a few minutes ago, and answered "1 day".

The tool returned `transit_days: 1`, and the answer reports exactly that: one day in transit from Denver to Boulder. That's not the pump's 1-day lead time at the Denver depot from the earlier runs. The question asked only about shipping, so the answer leaves the lead time out. In the trace, the tool's span is named `kestrel_estimate_delivery`. It's the same kind of span the agent's own tools made in earlier modules, so tracing didn't change when the tools moved.

The tool catalog now belongs to the platform, and it changed without a new release of the agent.

## If the result differs

If the address cell in step 2 says this environment has no lab MCP Gateway, your workbench wasn't set up with one. You can't fix that from the notebook, so ask for help.

If the check in step 2 prints `gateway_url: ''`, `config.toml` isn't saved with the address. Save it and run that cell again. If the cell fails with an error about TOML, the address is missing its quotes. Until that check shows the address, the later cells print "Set gateway_url first." and run nothing, and nothing gets registered.

If a run fails with HTTP 401 or 403, the gateway didn't accept the login your workbench sends. The notebook's steps use your workbench's own identity, not an admin login. That's an environment problem, not something in the agent, so ask for help.

Through the gateway, `kestrel_find_part` should return `KI-CP-40817`, and `kestrel_check_stock` at Denver should give quantity 3 and a lead time of 1 day. A made-up part number such as `KI-XX-00000` means the lookup broke between the two tools. An extra check of the seal kit is fine.

If `kestrel_estimate_delivery` isn't listed after you register it, wait a few seconds and run the registration cell again. If the first delivery run already used it, the tool was registered on an earlier attempt at this module. After registration, the delivery run should call `kestrel_estimate_delivery`, get `transit_days: 1`, and answer one day.